# Notebook 1: Graph Extraction and Knowledge Graph Construction

This notebook implements the graph construction stage of the project. The goal is to build a knowledge graph from one DPR Wikipedia shard, which will later be used in a separate notebook for graph-only retrieval and hybrid DPR + graph retrieval experiments.

The method is inspired by Microsoft GraphRAG, especially the idea of using an LLM to extract entities and relationships from text chunks and convert them into a graph index. In this notebook, the full Microsoft GraphRAG pipeline is not reproduced. Instead, only the graph extraction and graph construction stages are adapted for Wikipedia-based question answering.

## Main Goal

The main goal of this notebook is to transform unstructured Wikipedia text chunks into a structured graph representation.

The graph is built from:

* extracted entities,
* extracted relationships,
* original chunk information,
* original document information,
* textual evidence supporting each relationship.

## Workflow

The notebook follows this workflow:

1. Load text chunks from one DPR Wikipedia shard.
2. Apply an LLM-based entity and relationship extraction prompt.
3. Parse the extracted entities and relationships.
4. Save the extraction results in JSONL format.
5. Build a knowledge graph using `networkx.MultiDiGraph`.
6. Add document nodes, chunk nodes, and entity nodes.
7. Add edges connecting chunks to documents, chunks to entities, and entities to related entities.
8. Save the final graph for use in the retrieval notebook.

## Relation to GraphRAG

This notebook is GraphRAG-inspired because it follows the same general idea of building a graph index from text using LLM extraction. In GraphRAG, entities become graph nodes and relationships become graph edges. This project adapts that idea to a Wikipedia DPR shard.

However, the purpose here is different from the full GraphRAG paper. Microsoft GraphRAG focuses on global sensemaking and query-focused summarization over large corpora. In this project, the graph is mainly built as a retrieval structure for open-domain question answering.

## Prompting Strategy

The extraction prompt used in this notebook is a one-shot / few-shot structured extraction prompt. It includes one example to show the desired output format, so it is not zero-shot. It is also not Chain-of-Thought prompting, because the model is not asked to explain its reasoning step by step. Instead, the model directly outputs structured entity and relationship records.

The prompt is made conservative for retrieval purposes. It requires relationships to be explicitly supported by the text, avoids outside knowledge, prevents unsupported inference, and stores evidence for each extracted relationship.

## References

* Lewis et al. (2020), *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*
  https://arxiv.org/abs/2005.11401

* Edge et al. (2025), *From Local to Global: A GraphRAG Approach to Query-Focused Summarization*
  https://arxiv.org/abs/2404.16130

* Microsoft GraphRAG GitHub repository
  https://github.com/microsoft/graphrag

## Note

This notebook only performs graph extraction and graph construction. The DPR baseline, generator, graph retrieval, and hybrid retrieval experiments are implemented in a separate notebook.


In [11]:
pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [12]:
pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [13]:
!pip install -U datasets huggingface_hub

In [14]:
!pip install -U huggingface_hub datasets

In [15]:
# ============================================================
# DATA PREPARATION FOR FULL-SHARD GRAPH PIPELINE
# Load shard -> group passages by title -> chunk articles
# ============================================================

import pandas as pd
import hashlib
import re
import os



import tiktoken

# ------------------------------------------------------------
# 1. Config
# ------------------------------------------------------------

DATASET_PATH = (
    "hf://datasets/facebook/wiki_dpr/"
    "data/psgs_w100/"
    "nq/train-00000-of-00157.parquet"
)

OUTPUT_DIR = "graph_data_full_shard"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100

enc = tiktoken.get_encoding("cl100k_base")

# ------------------------------------------------------------
# 2. Load full shard
# ------------------------------------------------------------

print("\nLOADING DATASET")

df = pd.read_parquet(DATASET_PATH)

print("Raw dataframe shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

# Keep only needed columns
df = df[["id", "title", "text"]].copy()

df["title"] = df["title"].astype(str)
df["text"] = df["text"].astype(str)

print("\nFULL SHARD STATS")
print("Passages:", len(df))
print("Unique titles/articles:", df["title"].nunique())

# ------------------------------------------------------------
# 3. Group passages into one article per title
# ------------------------------------------------------------

print("\nGROUPING PASSAGES BY TITLE")

docs_one_row = (
    df.sort_values(["title", "id"])
      .groupby("title", as_index=False)
      .agg({"text": " ".join})
)

print("Documents/articles:", len(docs_one_row))
print(docs_one_row.head())

# ------------------------------------------------------------
# 4. Helper functions
# ------------------------------------------------------------

def slugify_title(title):
    title = str(title).strip().lower()
    title = re.sub(r"[^a-z0-9]+", "_", title)
    title = re.sub(r"_+", "_", title).strip("_")
    return title[:80]

def make_hash(text, n=10):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:n]

def make_document_id(title, doc_idx=None):
    slug = slugify_title(title)
    short_hash = make_hash(title, n=8)

    if doc_idx is not None:
        return f"doc_{doc_idx:06d}_{slug}_{short_hash}"

    return f"doc_{slug}_{short_hash}"

def make_text_unit_id(document_id, chunk_idx):
    return f"{document_id}_chunk_{chunk_idx:04d}"

def chunk_tokens(text, chunk_size=1000, overlap=100):
    tokens = enc.encode(str(text))
    step = chunk_size - overlap
    chunks = []

    for start in range(0, len(tokens), step):
        end = min(start + chunk_size, len(tokens))
        chunk_text = enc.decode(tokens[start:end]).strip()

        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "start_token": start,
                "end_token": end,
                "n_tokens": end - start
            })

        if end == len(tokens):
            break

    return chunks

# ------------------------------------------------------------
# 5. Chunk grouped articles into text units
# ------------------------------------------------------------

print("\nCHUNKING ARTICLES")

text_units = []

for doc_number, (_, row) in enumerate(docs_one_row.iterrows()):
    title = row["title"]
    document_id = make_document_id(title, doc_number)

    chunks = chunk_tokens(
        row["text"],
        chunk_size=CHUNK_SIZE,
        overlap=CHUNK_OVERLAP
    )

    for chunk_idx, ch in enumerate(chunks):
        text_unit_id = make_text_unit_id(document_id, chunk_idx)

        text_units.append({
            "text_unit_id": text_unit_id,
            "chunk_id": text_unit_id,
            "document_id": document_id,
            "document_index": doc_number,
            "title": title,
            "chunk_index": chunk_idx,
            "text": ch["text"],
            "n_tokens": ch["n_tokens"],
            "start_token": ch["start_token"],
            "end_token": ch["end_token"]
        })

text_units_df = pd.DataFrame(text_units)

text_units_df["chunk_uid"] = (
    text_units_df["document_id"].astype(str)
    + "_"
    + text_units_df["chunk_index"].astype(str)
)

print("\nFINAL DATA STATS")
print("Documents:", text_units_df["document_id"].nunique())
print("Chunks/text units:", len(text_units_df))
print("Average tokens per chunk:", round(text_units_df["n_tokens"].mean(), 2))
print("Max tokens per chunk:", text_units_df["n_tokens"].max())

display(text_units_df[["chunk_id", "title", "chunk_index", "n_tokens", "text"]].head())

# ------------------------------------------------------------
# 6. Save prepared data
# ------------------------------------------------------------

docs_path = os.path.join(OUTPUT_DIR, "wiki_full_shard_documents.parquet")
chunks_path = os.path.join(OUTPUT_DIR, "wiki_full_shard_text_units.parquet")

docs_one_row.to_parquet(docs_path, index=False)
text_units_df.to_parquet(chunks_path, index=False)

print("\nSAVED FILES")
print("Documents saved to:", docs_path)
print("Text units saved to:", chunks_path)


LOADING DATASET
Raw dataframe shape: (133856, 4)
Columns: ['id', 'text', 'title', 'embeddings']
  id                                               text  title  \
0  1  Aaron Aaron ( or ; "Ahärôn") is a prophet, hig...  Aaron   
1  2  God at Sinai granted Aaron the priesthood for ...  Aaron   
2  3  his rod turn into a snake. Then he stretched o...  Aaron   
3  4  however, Aaron and Hur remained below to look ...  Aaron   
4  5  Aaron and his sons to the priesthood, and arra...  Aaron   

                                          embeddings  
0  [0.013342111, 0.58217376, -0.31309745, -0.6991...  
1  [-0.19236332, 0.539003, -0.5652932, -0.5195250...  
2  [-0.23045847, 0.28877887, -0.3449004, -0.14077...  
3  [0.107315615, 0.5992388, -0.37498242, -0.53419...  
4  [0.32623303, 0.51600194, -0.5568064, -0.494033...  

FULL SHARD STATS
Passages: 133856
Unique titles/articles: 4352

GROUPING PASSAGES BY TITLE
Documents/articles: 4352
                                    title  \
0         "...

,chunk_id,title,chunk_index,n_tokens,text
0,doc_000000_baby_one_more_time_album_39167545_c...,"""...Baby One More Time (album)""",0,1000,...Baby One More Time (album) ...Baby One More...
1,doc_000000_baby_one_more_time_album_39167545_c...,"""...Baby One More Time (album)""",1,1000,"Pop"", ""Thinkin' About You"", and ""You Got It Al..."
2,doc_000000_baby_one_more_time_album_39167545_c...,"""...Baby One More Time (album)""",2,1000,as noted by Kyle Anderson of MTV. The eleventh...
3,doc_000000_baby_one_more_time_album_39167545_c...,"""...Baby One More Time (album)""",3,1000,"Nigel Dick, portrays Spears as a student from ..."
4,doc_000000_baby_one_more_time_album_39167545_c...,"""...Baby One More Time (album)""",4,1000,I-Zone as the tour's official camera. Spears u...



SAVED FILES
Documents saved to: graph_data_full_shard/wiki_full_shard_documents.parquet
Text units saved to: graph_data_full_shard/wiki_full_shard_text_units.parquet


In [16]:
num_titles = df["title"].nunique()
num_passages = len(df)
num_documents = len(docs_one_row)
num_chunks = len(text_units_df)

print("\n===== FINAL SUMMARY =====")
print("Unique titles/articles:", num_titles)
print("Original passages:", num_passages)
print("Grouped documents:", num_documents)
print("Final chunks/text units:", num_chunks)

print("\nChunks per document:")
print(text_units_df.groupby("document_id").size().describe())

print("\nTokens per chunk:")
print(text_units_df["n_tokens"].describe())


===== FINAL SUMMARY =====
Unique titles/articles: 4352
Original passages: 133856
Grouped documents: 4352
Final chunks/text units: 21625

Chunks per document:
count    4352.000000
mean        4.968980
std         4.383344
min         1.000000
25%         2.000000
50%         3.000000
75%         7.000000
max        31.000000
dtype: float64

Tokens per chunk:
count    21625.000000
mean       910.629780
std        212.221279
min        101.000000
25%       1000.000000
50%       1000.000000
75%       1000.000000
max       1000.000000
Name: n_tokens, dtype: float64


## Graph Extraction Prompt

This section implements the graph extraction stage used to build the graph-based retrieval component. The method is inspired by Microsoft GraphRAG:

* Paper: *From Local to Global: A GraphRAG Approach to Query-Focused Summarization*
  https://arxiv.org/abs/2404.16130

* GitHub repository: Microsoft GraphRAG
  https://github.com/microsoft/graphrag

In the GraphRAG pipeline, source documents are divided into text chunks, and an LLM is used to extract entities and relationships from each chunk. The extracted entities can be represented as graph nodes, while the relationships can be represented as graph edges. This project adapts that idea to a Wikipedia DPR shard in order to support graph-based retrieval and hybrid DPR + graph retrieval.

The prompt used here is best described as a **GraphRAG  one-shot structured extraction prompt**. It is **one-shot** because the prompt includes one example showing the desired entity and relationship output format. It can also be described more generally as **few-shot prompting**, since few-shot prompting refers to guiding the model with one or more examples. 

Compared with the original GraphRAG-style extraction prompt, this version is adapted for retrieval over Wikipedia text. The prompt requires relationships to be explicitly supported by the text, prevents the use of outside knowledge, avoids creating edges from simple co occurrence, and stores the supporting evidence for each relationship. This makes the extracted graph more conservative and more suitable for question answering, where unsupported graph edges could lead to incorrect retrieval.

The extraction produces two main record types:

1. **Entities**
   Each entity contains:

   * entity name
   * entity type
   * entity description

2. **Relationships**
   Each relationship contains:

   * source entity
   * target entity
   * relationship description
   * relationship strength
   * supporting evidence from the original chunk

A second relationship review pass is also applied. This pass does not create new entities. Instead, it only checks whether there are missing relationships among the already extracted entities. This is intended to improve relationship recall while keeping the graph grounded in the original Wikipedia chunk.

### Adaptation from GraphRAG microsoft paper

The prompt follows the same general idea as GraphRAG: using an LLM to extract entities and relationships from text chunks. However, it is adapted for Wikipedia-based retrieval rather than global corpus summarization.

Compared with the original GraphRAG-style extraction prompt, this version is more conservative. It includes rules that:

prevent the model from using outside knowledge,
prevent unsupported inference,
prevent creating relationships only from entity co-occurrence,
require each relationship to be explicitly supported by the text,
store the supporting evidence for each extracted relationship,
include the article title or main topic as an entity when it is discussed in the chunk.

This makes the extracted graph more suitable for question answering, because unsupported or hallucinated graph edges could lead to incorrect retrieval.


In [ ]:
import json
from openai import OpenAI
import json
from collections import defaultdict



# ============================================================
# API CLIENT
# ============================================================

os.environ["OPENROUTER_API_KEY"] = "################################################################"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

MODEL = "openai/gpt-4o-mini"


# ============================================================
# CONFIG
# ============================================================

ENTITY_TYPES = [
    "PERSON",
    "ORGANIZATION",
    "GEO",
    "EVENT",
    "WORK",
    "CONCEPT",
    "DATE",
    "GROUP",
    "OTHER"
]


# ============================================================
# PROMPTS
# ============================================================

GRAPH_EXTRACTION_PROMPT = """
-Goal-
Given a Wikipedia text chunk and a list of entity types, extract:
1. important entities
2. explicitly supported relationships among those entities

This graph will be used for retrieval over Wikipedia text.

Entity types:
{entity_types}

Rules:
- Always include the article title/main topic as an entity if discussed in the text.
- Extract entities useful for retrieval.
- Do not extract random names that are not useful.
- Do not use outside knowledge.
- Do not infer unsupported facts.

Entity format:
("entity"<|><entity_name><|><entity_type><|><entity_description>)

Relationship rules:
- Extract only relationships explicitly stated in the text.
- Do not create a relationship only because two entities co-occur.
- The evidence must directly support the relationship.
- Prefer conservative wording.
- Closely paraphrase the evidence.
- Favor precision over recall.

Relationship format:
("relationship"<|><source_entity><|><target_entity><|><relationship_description><|><relationship_strength><|><evidence>)

Relationship strength:
1-3 = weak
4-6 = moderate
7-10 = strong

Use ## as the delimiter.
End with <|COMPLETE|>.

###################### EXAMPLES ######################

Example:
Title: Anime
Text:
Anime is distributed through television broadcasts, films, and streaming platforms such as Crunchyroll and Netflix.
The 1988 film Akira is largely credited with popularizing anime in the Western world.

Output:
("entity"<|>ANIME<|>CONCEPT<|>Anime is a form of media distributed through broadcasts, films, and streaming platforms)##
("entity"<|>CRUNCHYROLL<|>ORGANIZATION<|>Crunchyroll is a streaming platform through which anime is distributed)##
("entity"<|>NETFLIX<|>ORGANIZATION<|>Netflix is a streaming platform through which anime is distributed)##
("entity"<|>AKIRA<|>WORK<|>Akira is a film credited with popularizing anime in the Western world)##
("relationship"<|>CRUNCHYROLL<|>ANIME<|>Crunchyroll is a streaming platform through which anime is distributed<|>8<|>Anime is distributed through television broadcasts, films, and streaming platforms such as Crunchyroll and Netflix)##
("relationship"<|>NETFLIX<|>ANIME<|>Netflix is a streaming platform through which anime is distributed<|>8<|>Anime is distributed through television broadcasts, films, and streaming platforms such as Crunchyroll and Netflix)##
("relationship"<|>AKIRA<|>ANIME<|>Akira is credited with popularizing anime in the Western world<|>9<|>The 1988 film Akira is largely credited with popularizing anime in the Western world)<|COMPLETE|>

###################### REAL DATA ######################

Title: {title}
Chunk_id: {chunk_id}
Text:
{input_text}

Output:
"""


RELATIONSHIP_REVIEW_PROMPT = """
You are reviewing a knowledge graph extraction for Wikipedia retrieval.

Original title:
{title}

Original chunk id:
{chunk_id}

Original text:
{input_text}

Previously extracted entities:
{entities}

Previously extracted relationships:
{relationships}

Task:
Find missing relationships among the previously extracted entities.

Rules:
- Do NOT extract new entities.
- Only use entities from the entity list.
- Add only NEW relationships.
- Do NOT repeat existing relationships.
- Every relationship must be explicitly supported by the original text.
- Do NOT use outside knowledge.
- Do NOT infer relationships from co-occurrence.
- If evidence is weak or unclear, do not add the relationship.
- Prefer relationships involving the main topic/title entity when clearly supported.
- Use conservative wording.
- Closely paraphrase the evidence.

Output format:
("relationship"<|><source_entity><|><target_entity><|><relationship_description><|><relationship_strength><|><evidence>)

Use ## as the delimiter.
End with <|COMPLETE|>.
"""


# ============================================================
# LLM CALL
# ============================================================

def call_llm(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        max_tokens=1200,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content


# ============================================================
# DEDUPLICATION
# ============================================================

def deduplicate_entities(entities):
    seen = set()
    output = []

    for e in entities:
        key = (
            e["chunk_id"],
            e["entity_name"].strip().lower(),
            e["entity_type"].strip().lower()
        )

        if key not in seen:
            seen.add(key)
            output.append(e)

    return output


def deduplicate_relationships(relationships):
    seen = set()
    output = []

    for r in relationships:
        key = (
            r["chunk_id"],
            r["source_entity"].strip().lower(),
            r["target_entity"].strip().lower(),
            r["relationship_description"].strip().lower()
        )

        if key not in seen:
            seen.add(key)
            output.append(r)

    return output


def relationship_key(r):
    return (
        r["chunk_id"],
        r["source_entity"].strip().lower(),
        r["target_entity"].strip().lower(),
        r["relationship_description"].strip().lower()
    )


def merge_relationships(existing_relationships, new_relationships):
    seen = set(relationship_key(r) for r in existing_relationships)
    merged = list(existing_relationships)

    for r in new_relationships:
        key = relationship_key(r)

        if key not in seen:
            seen.add(key)
            merged.append(r)

    return merged


# ============================================================
# PARSING
# ============================================================

def clean_record(record):
    return record.replace("<|COMPLETE|>", "").strip()


def parse_graphrag_output(raw_output, title="Wikipedia Article", chunk_id="chunk_0"):
    raw_output = raw_output.replace("<|COMPLETE|>", "")

    records = [
        clean_record(r)
        for r in raw_output.split("##")
        if clean_record(r)
    ]

    entities = []
    relationships = []

    for record in records:

        if record.startswith('("entity"<|>'):
            content = record[len('("entity"<|>'):]

            if content.endswith(")"):
                content = content[:-1]

            parts = [p.strip() for p in content.split("<|>")]

            if len(parts) >= 3:
                entities.append({
                    "chunk_id": chunk_id,
                    "title": title,
                    "entity_name": parts[0],
                    "entity_type": parts[1],
                    "entity_description": parts[2],
                })

        elif record.startswith('("relationship"<|>'):
            content = record[len('("relationship"<|>'):]

            if content.endswith(")"):
                content = content[:-1]

            parts = [p.strip() for p in content.split("<|>")]

            if len(parts) >= 5:
                try:
                    strength = int(parts[3])
                except Exception:
                    strength = None

                relationships.append({
                    "chunk_id": chunk_id,
                    "title": title,
                    "source_entity": parts[0],
                    "target_entity": parts[1],
                    "relationship_description": parts[2],
                    "relationship_strength": strength,
                    "evidence": parts[4],
                })

    return {
        "entities": deduplicate_entities(entities),
        "relationships": deduplicate_relationships(relationships)
    }


def parse_relationship_only_output(raw_output, title="Wikipedia Article", chunk_id="chunk_0"):
    parsed = parse_graphrag_output(
        raw_output=raw_output,
        title=title,
        chunk_id=chunk_id
    )

    return parsed["relationships"]


# ============================================================
# EXTRACTION PASSES
# ============================================================

def run_first_pass(input_text, title="Wikipedia Article", chunk_id="chunk_0"):
    prompt = GRAPH_EXTRACTION_PROMPT.format(
        entity_types=", ".join(ENTITY_TYPES),
        title=title,
        chunk_id=chunk_id,
        input_text=input_text
    )

    return call_llm(prompt)


def run_relationship_review_pass(parsed, input_text, title="Wikipedia Article", chunk_id="chunk_0"):
    entities_text = json.dumps(
        parsed["entities"],
        indent=2,
        ensure_ascii=False
    )

    relationships_text = json.dumps(
        parsed["relationships"],
        indent=2,
        ensure_ascii=False
    )

    prompt = RELATIONSHIP_REVIEW_PROMPT.format(
        title=title,
        chunk_id=chunk_id,
        input_text=input_text,
        entities=entities_text,
        relationships=relationships_text
    )

    return call_llm(prompt)


def extract_graph_for_wiki_chunk(
    input_text,
    title="Wikipedia Article",
    chunk_id="chunk_0",
    use_relationship_review=True
):
    raw_first = run_first_pass(
        input_text=input_text,
        title=title,
        chunk_id=chunk_id
    )

    parsed = parse_graphrag_output(
        raw_output=raw_first,
        title=title,
        chunk_id=chunk_id
    )

    raw_review = ""

    if use_relationship_review and len(parsed["entities"]) > 1:
        raw_review = run_relationship_review_pass(
            parsed=parsed,
            input_text=input_text,
            title=title,
            chunk_id=chunk_id
        )

        review_relationships = parse_relationship_only_output(
            raw_output=raw_review,
            title=title,
            chunk_id=chunk_id
        )

        parsed["relationships"] = merge_relationships(
            parsed["relationships"],
            review_relationships
        )

    return raw_first, raw_review, parsed


# ============================================================
# TEST
# ============================================================

sample_text = """
Anime is hand-drawn and computer-generated animation originating from Japan.
The earliest commercial Japanese animations date to 1917.
Osamu Tezuka helped popularize anime through works such as Astro Boy.
Anime is distributed through television broadcasts, films, home media, and streaming services such as Crunchyroll and Netflix.
The 1988 film Akira is largely credited with popularizing anime in the Western world during the early 1990s.
"""

raw_first, raw_review, parsed_graph = extract_graph_for_wiki_chunk(
    input_text=sample_text,
    title="Anime",
    chunk_id="anime_test_0",
    use_relationship_review=True
)

print("RAW FIRST PASS")
print(raw_first)

print("\nRAW RELATIONSHIP REVIEW PASS")
print(raw_review)

print("\nPARSED GRAPH")
print(json.dumps(parsed_graph, indent=2, ensure_ascii=False))

print("\nEntity count:", len(parsed_graph["entities"]))
print("Relationship count:", len(parsed_graph["relationships"]))

RAW FIRST PASS
("entity"<|>ANIME<|>CONCEPT<|>Anime is hand-drawn and computer-generated animation originating from Japan)##
("entity"<|>OSAMU TEZUKA<|>PERSON<|>Osamu Tezuka helped popularize anime through works such as Astro Boy)##
("entity"<|>ASTRO BOY<|>WORK<|>Astro Boy is a work by Osamu Tezuka that helped popularize anime)##
("entity"<|>CRUNCHYROLL<|>ORGANIZATION<|>Crunchyroll is a streaming service through which anime is distributed)##
("entity"<|>NETFLIX<|>ORGANIZATION<|>Netflix is a streaming service through which anime is distributed)##
("entity"<|>AKIRA<|>WORK<|>Akira is a film credited with popularizing anime in the Western world)##
("entity"<|>1917<|>DATE<|>The earliest commercial Japanese animations date to 1917)##
("relationship"<|>OSAMU TEZUKA<|>ANIME<|>Osamu Tezuka helped popularize anime<|>7<|>Osamu Tezuka helped popularize anime through works such as Astro Boy)##
("relationship"<|>ASTRO BOY<|>ANIME<|>Astro Boy is a work that contributed to the popularity of anime<|>6<|

In [19]:
row = text_units_df.iloc[0]

raw_first, raw_review, parsed_graph = extract_graph_for_wiki_chunk(
    input_text=row["text"],
    title=row["title"],
    chunk_id=row["chunk_id"],
    use_relationship_review=False
)

print(json.dumps(parsed_graph, indent=2))

{
  "entities": [
    {
      "chunk_id": "doc_000000_baby_one_more_time_album_39167545_chunk_0000",
      "title": "\"...Baby One More Time (album)\"",
      "entity_name": "...BABY ONE MORE TIME",
      "entity_type": "WORK",
      "entity_description": "...Baby One More Time is the debut studio album by American pop singer Britney Spears"
    },
    {
      "chunk_id": "doc_000000_baby_one_more_time_album_39167545_chunk_0000",
      "title": "\"...Baby One More Time (album)\"",
      "entity_name": "BRITNEY SPEARS",
      "entity_type": "PERSON",
      "entity_description": "Britney Spears is an American pop singer and the artist behind the album ...Baby One More Time"
    },
    {
      "chunk_id": "doc_000000_baby_one_more_time_album_39167545_chunk_0000",
      "title": "\"...Baby One More Time (album)\"",
      "entity_name": "JANUARY 12, 1999",
      "entity_type": "DATE",
      "entity_description": "The album ...Baby One More Time was released on January 12, 1999"
    },
    {

In [10]:
for i in range(10):
    row = text_units_df.iloc[i]

    _, _, parsed_graph = extract_graph_for_wiki_chunk(
        input_text=row["text"],
        title=row["title"],
        chunk_id=row["chunk_id"],
        use_relationship_review=False
    )

    print(
        i,
        len(parsed_graph["entities"]),
        len(parsed_graph["relationships"])
    )

0 15 11
1 10 9
2 14 10
3 14 10
4 6 5
5 14 10
6 8 7
7 13 10
8 17 8
9 8 6


In [ ]:
num_chunks = len(text_units_df)

print("Estimated entities:", int(12.8 * num_chunks))
print("Estimated relationships:", int(10.4 * num_chunks))

# full safe extraciton

In [20]:
# ============================================================
# FULL RESUMABLE GRAPH EXTRACTION
# SAFE FOR LONG RUNS
# ============================================================


import time
from tqdm.auto import tqdm

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

EXTRACTION_DIR = "graph_extraction_checkpoint"
os.makedirs(EXTRACTION_DIR, exist_ok=True)

OUTPUT_JSONL = os.path.join(
    EXTRACTION_DIR,
    "graph_extraction_results.jsonl"
)

FAILED_JSONL = os.path.join(
    EXTRACTION_DIR,
    "graph_extraction_failed.jsonl"
)

PROGRESS_FILE = os.path.join(
    EXTRACTION_DIR,
    "processed_chunk_ids.txt"
)

USE_RELATIONSHIP_REVIEW = False

SLEEP_BETWEEN_CALLS = 0.5

MAX_RETRIES = 3
RETRY_SLEEP = 10


# ------------------------------------------------------------
# FILE HELPERS
# ------------------------------------------------------------

def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )
        f.flush()
        os.fsync(f.fileno())


def append_processed_chunk(chunk_id):
    with open(PROGRESS_FILE, "a", encoding="utf-8") as f:
        f.write(str(chunk_id) + "\n")
        f.flush()
        os.fsync(f.fileno())


def load_processed_chunks():

    if not os.path.exists(PROGRESS_FILE):
        return set()

    with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
        return set(
            line.strip()
            for line in f
            if line.strip()
        )


def count_jsonl_lines(path):

    if not os.path.exists(path):
        return 0

    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)


# ------------------------------------------------------------
# EXTRACTION WITH RETRIES
# ------------------------------------------------------------

def extract_one_chunk_with_retry(row):

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            raw_first, raw_review, parsed_graph = (
                extract_graph_for_wiki_chunk(
                    input_text=row["text"],
                    title=row["title"],
                    chunk_id=row["chunk_id"],
                    use_relationship_review=USE_RELATIONSHIP_REVIEW
                )
            )

            return {
                "chunk_id": row["chunk_id"],
                "document_id": row["document_id"],
                "title": row["title"],
                "chunk_index": int(row["chunk_index"]),
                "n_tokens": int(row["n_tokens"]),

                "entity_count":
                    len(parsed_graph["entities"]),

                "relationship_count":
                    len(parsed_graph["relationships"]),

                "entities":
                    parsed_graph["entities"],

                "relationships":
                    parsed_graph["relationships"],

                "status": "ok",
                "error": ""
            }

        except Exception as e:

            last_error = str(e)

            print(
                f"\nAttempt {attempt}/{MAX_RETRIES}"
                f" failed for {row['chunk_id']}"
            )

            print(last_error)

            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP * attempt)

    return {
        "chunk_id": row["chunk_id"],
        "document_id": row["document_id"],
        "title": row["title"],
        "chunk_index": int(row["chunk_index"]),
        "n_tokens": int(row["n_tokens"]),
        "entity_count": 0,
        "relationship_count": 0,
        "entities": [],
        "relationships": [],
        "status": "failed",
        "error": last_error
    }


# ------------------------------------------------------------
# LOAD PREVIOUS PROGRESS
# ------------------------------------------------------------

processed_chunks = load_processed_chunks()

todo = text_units_df[
    ~text_units_df["chunk_id"]
        .astype(str)
        .isin(processed_chunks)
]

print("=" * 60)
print("TOTAL CHUNKS:", len(text_units_df))
print("ALREADY PROCESSED:", len(processed_chunks))
print("REMAINING:", len(todo))
print("=" * 60)


# ------------------------------------------------------------
# MAIN LOOP
# ------------------------------------------------------------

total_entities = 0
total_relationships = 0
total_failed = 0

start_time = time.time()

pbar = tqdm(
    total=len(todo),
    desc="Graph Extraction",
    unit="chunk"
)

for _, row in todo.iterrows():

    row = row.to_dict()

    result = extract_one_chunk_with_retry(row)

    if result["status"] == "ok":

        append_jsonl(
            OUTPUT_JSONL,
            result
        )

        total_entities += result["entity_count"]

        total_relationships += (
            result["relationship_count"]
        )

    else:

        append_jsonl(
            FAILED_JSONL,
            result
        )

        total_failed += 1

    append_processed_chunk(
        result["chunk_id"]
    )

    pbar.update(1)

    processed_now = pbar.n

    pbar.set_postfix(
        entities=total_entities,
        relations=total_relationships,
        failed=total_failed,
        avg_ent=round(
            total_entities /
            max(1, processed_now),
            1
        ),
        avg_rel=round(
            total_relationships /
            max(1, processed_now),
            1
        )
    )

    time.sleep(
        SLEEP_BETWEEN_CALLS
    )

pbar.close()

elapsed_hours = (
    time.time() - start_time
) / 3600

print("\n" + "=" * 60)
print("EXTRACTION COMPLETE")
print("=" * 60)

print(
    "Successful:",
    count_jsonl_lines(OUTPUT_JSONL)
)

print(
    "Failed:",
    count_jsonl_lines(FAILED_JSONL)
)

print(
    "Elapsed hours:",
    round(elapsed_hours, 2)
)

print(
    "Results file:",
    OUTPUT_JSONL
)

print(
    "Failed file:",
    FAILED_JSONL
)

print(
    "Progress file:",
    PROGRESS_FILE
)

TOTAL CHUNKS: 21625
ALREADY PROCESSED: 21625
REMAINING: 0


Graph Extraction: 0chunk [00:00, ?chunk/s]


EXTRACTION COMPLETE
Successful: 18964
Failed: 4577
Elapsed hours: 0.0
Results file: graph_extraction_checkpoint/graph_extraction_results.jsonl
Failed file: graph_extraction_checkpoint/graph_extraction_failed.jsonl
Progress file: graph_extraction_checkpoint/processed_chunk_ids.txt


# Graph Construction using **networkx.MultiDiGraph.**

In [21]:
# ============================================================
# BUILD GRAPH ONLY
#========================================================


import networkx as nx

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

RESULTS_JSONL = "graph_extraction_checkpoint/graph_extraction_results.jsonl"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def read_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def canonical_entity_name(name):
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return name.upper()


def entity_node_id(name):
    return "entity::" + canonical_entity_name(name)


def chunk_node_id(chunk_id):
    return "chunk::" + str(chunk_id)


def document_node_id(document_id):
    return "doc::" + str(document_id)


def safe_text(x):
    if x is None:
        return ""
    return str(x)


def safe_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default


def safe_float(x, default=1.0):
    try:
        return float(x)
    except Exception:
        return default


# ------------------------------------------------------------
# LOAD EXTRACTION RESULTS
# ------------------------------------------------------------

records = read_jsonl(RESULTS_JSONL)

print("Loaded records:", len(records))


# ------------------------------------------------------------
# BUILD GRAPH
# ------------------------------------------------------------

G = nx.MultiDiGraph()

for record in records:

    chunk_id = safe_text(record.get("chunk_id"))
    document_id = safe_text(record.get("document_id"))
    title = safe_text(record.get("title"))
    chunk_index = safe_int(record.get("chunk_index"))
    n_tokens = safe_int(record.get("n_tokens"))

    chunk_node = chunk_node_id(chunk_id)
    doc_node = document_node_id(document_id)

    # --------------------------------------------------------
    # Add document node
    # --------------------------------------------------------

    if not G.has_node(doc_node):
        G.add_node(
            doc_node,
            node_type="document",
            document_id=document_id,
            title=title,
            label=title
        )

    # --------------------------------------------------------
    # Add chunk node
    # --------------------------------------------------------

    if not G.has_node(chunk_node):
        G.add_node(
            chunk_node,
            node_type="chunk",
            chunk_id=chunk_id,
            document_id=document_id,
            title=title,
            chunk_index=chunk_index,
            n_tokens=n_tokens,
            label=chunk_id
        )

    # --------------------------------------------------------
    # Link chunk to document
    # --------------------------------------------------------

    G.add_edge(
        chunk_node,
        doc_node,
        edge_type="BELONGS_TO",
        weight=1.0
    )

    # --------------------------------------------------------
    # Add entity nodes and chunk -> entity edges
    # --------------------------------------------------------

    for ent in record.get("entities", []):

        name = ent.get("entity_name", "")
        entity_type = ent.get("entity_type", "OTHER")
        description = ent.get("entity_description", "")

        ent_node = entity_node_id(name)

        if not G.has_node(ent_node):
            G.add_node(
                ent_node,
                node_type="entity",
                entity_name=canonical_entity_name(name),
                entity_type=entity_type,
                description=description,
                label=canonical_entity_name(name),
                mention_count=0
            )

        G.nodes[ent_node]["mention_count"] += 1

        # chunk mentions entity
        G.add_edge(
            chunk_node,
            ent_node,
            edge_type="MENTIONS",
            weight=1.0,
            chunk_id=chunk_id,
            document_id=document_id,
            title=title
        )

    # --------------------------------------------------------
    # Add entity -> entity relationship edges
    # --------------------------------------------------------

    for rel in record.get("relationships", []):

        source = rel.get("source_entity", "")
        target = rel.get("target_entity", "")

        source_node = entity_node_id(source)
        target_node = entity_node_id(target)

        # Add missing source node if needed
        if not G.has_node(source_node):
            G.add_node(
                source_node,
                node_type="entity",
                entity_name=canonical_entity_name(source),
                entity_type="OTHER",
                description="",
                label=canonical_entity_name(source),
                mention_count=0
            )

        # Add missing target node if needed
        if not G.has_node(target_node):
            G.add_node(
                target_node,
                node_type="entity",
                entity_name=canonical_entity_name(target),
                entity_type="OTHER",
                description="",
                label=canonical_entity_name(target),
                mention_count=0
            )

        relationship_description = safe_text(
            rel.get("relationship_description", "")
        )

        relationship_strength = safe_float(
            rel.get("relationship_strength", 1.0)
        )

        evidence = safe_text(
            rel.get("evidence", "")
        )

        G.add_edge(
            source_node,
            target_node,
            edge_type="RELATIONSHIP",
            relationship_description=relationship_description,
            weight=relationship_strength,
            evidence=evidence,
            chunk_id=chunk_id,
            document_id=document_id,
            title=title
        )


# ------------------------------------------------------------
# CHECK GRAPH
# ------------------------------------------------------------

print("Graph built successfully.")
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())





# ============================================================
# SAVE GRAPH FIXED VERSION
# ============================================================


import pickle


GRAPH_DIR = "graph_extraction_checkpoint/built_graph"
os.makedirs(GRAPH_DIR, exist_ok=True)

GRAPHML_PATH = os.path.join(GRAPH_DIR, "knowledge_graph.graphml")
GEXF_PATH = os.path.join(GRAPH_DIR, "knowledge_graph.gexf")
PICKLE_PATH = os.path.join(GRAPH_DIR, "knowledge_graph.pkl")

# Optional: for Gephi / Cytoscape
nx.write_graphml(G, GRAPHML_PATH)
print("Saved GraphML:", GRAPHML_PATH)

# Optional: also for Gephi
nx.write_gexf(G, GEXF_PATH)
print("Saved GEXF:", GEXF_PATH)

# Best for loading back into Python
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

print("Saved Python graph:", PICKLE_PATH)



# import pickle

# PICKLE_PATH = "graph_extraction_checkpoint/built_graph/knowledge_graph.pkl"

# with open(PICKLE_PATH, "rb") as f: G = pickle.load(f)

# print("Loaded graph.") print("Nodes:", G.number_of_nodes()) print("Edges:", G.number_of_edges())

# To check whether the files were saved:

# import os

# for path in [GRAPHML_PATH, GEXF_PATH, PICKLE_PATH]: print(path, os.path.exists(path), os.path.getsize(path) if os.path.exists(path) else 0)

# Click to add a cell.



Loaded records: 18964
Graph built successfully.
Number of nodes: 148228
Number of edges: 414522
Saved GraphML: graph_extraction_checkpoint/built_graph/knowledge_graph.graphml
Saved GEXF: graph_extraction_checkpoint/built_graph/knowledge_graph.gexf
Saved Python graph: graph_extraction_checkpoint/built_graph/knowledge_graph.pkl
